In [ ]:
# Import the usual libraries and variables.
%run standard_imports.py

In [ ]:
# Get combined install counts for all of developers' apps.

db = Db("~/.clover/p801.cfg")
query = """
    SELECT
        developer.name,
        developer.uuid,
        SUM(IF(merchant_app.deleted_time IS NULL, 1, 0)) AS "install_count"
    FROM merchant_app
        INNER JOIN merchant ON merchant.id = merchant_app.merchant_id
        INNER JOIN merchant_gateway ON merchant_gateway.merchant_id = merchant.id
        INNER JOIN developer_app ON developer_app.id = merchant_app.app_id
        INNER JOIN developer ON developer.id = developer_app.developer_id
        INNER JOIN account ON account.id = developer.owner_account_id
    WHERE merchant_gateway.payment_processor_id != 3
    AND merchant.infolease_suppress_billing != 1
    AND""" + NOT_LIKE_DEVELOPER_NAMES_ACCOUNT_EMAILS + """
    AND developer.uuid NOT IN ("R5C8Q9XE2JD5R") /* Bypass */
    GROUP BY developer.id
    HAVING SUM(IF(merchant_app.deleted_time IS NULL, 1, 0)) > 0
    ORDER BY install_count DESC
    """
df = pd.read_sql(query, con=db.conn)
db.close()

df

In [ ]:
# Plot combined install counts as a histogram.

plt.hist(df['install_count'].dropna(), color='green', edgecolor='black', bins=50)
plt.title("Histogram of Developers' Combined Install Counts")
plt.xlabel('Combined Install Counts')
plt.ylabel('Developers')

df.describe()

In [ ]:
# Which developers are in the top 5% by combined install count?

df['install_count_rank'] = df['install_count'].rank(pct=True)
df[df['install_count'] > df['install_count'].quantile(.95)]